# Day 69：模拟面试日：技术面 + HR 面全真模拟评估打分器与 7 天改进清单生成（Jupyter Notebook 实战手册）

> **所属阶段**：第 6 阶段 · 第六阶段：毕业设计与就业冲刺 ｜ **第 10 周**：毕业答辩、简历作品集与面试冲刺结业  
> **GitLab 工单**：`STARMIND-069` ｜ **开发分支**：`feat/day69-mock-interview-coach`  
> **配套 HTML5 课件**：`courseware/day69/index.html` ｜ **独立版本快照**：`versions/day69/`

---

## 🎙️ 老陈开场旁白

早啊各位！今天是全真模拟面试日！考前不模拟，上场两行泪。很多同学技术面答得热火朝天，结果栽在 HR 面的'你为什么离职/转行？''你的期望薪资怎么算的？''你最大的缺点是什么？'或者在技术面里不知道自己到底哪一维拖了后腿。今天咱们把大厂真实的 **招聘委员会（Hiring Committee）评估表** 搬进代码里：一面技术基础与手撕、二面项目深度与系统设计、HR 面沟通与动机契合度，跑出明确的录用评级（`Strong Hire` / `Hire` / `Lean No Hire`），并且针对每一个低于 80 分的薄弱维度，自动生成一份精准到天的 7 天冲刺整改清单！

### 承上启下
- **昨日基线**：前两天我们系统梳理了一面高频手撕与二面系统设计，今天进入实战演练——技术一面 + 技术二面 + HR 面全真模拟评估与复盘！
- **明日预告**：明天（Day 70）就是盛大的结业日：前沿趋势展望（多模态 Agent / 具身智能 / MCP 生态）、高级 AI 工程师成长路线图，以及星辰智脑 v1.0.0 正式发布验收！


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day69'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-069）

学员在正式投递冲刺前需经历贴近国内大厂真实标准的'技术一面 + 技术二面 + HR 面'三轮全真模拟，并获得客观的录用定级与可执行改进计划。

**今日迭代目标：**
- 实现大厂招聘委员会风格三轮加权打分器（含一票否决项检查：如编码红线或诚信/价值观红线）。
- 实现自动化 7 天靶向改进计划生成器：按薄弱维度优先级匹配对应刷题、ADR 补强与 HR 话术演练任务。

**验收标准：**
- 打分器准确计算综合得分与录用决策（`STRONG_HIRE` / `HIRE` / `NO_HIRE`），且对单项跌破红线（<60分）触发降级拦截。
- 改进清单生成器能自动识别所有 `< 80` 分维度并按分数从低到高排布 7 天突击任务。
- 纯 Python 离线运行，pytest 100% 通过。

---

## 📚 课堂核心技术笔记

### 1. HR 面必考的 3 个高情商反问问题

1) '这个岗位入职前 3 个月，团队最期待我快速拿下的核心交付目标是什么？'（展现结果导向）；2) '目前团队在 AI Agent / RAG 落地中最头疼的工程瓶颈是什么？'（展现解决痛点意识）；3) '团队目前的模型评测与上线发布节奏是怎样的？'（展现成熟工程视角）。

> **⚠️ 老陈避坑**：千万不要反问'你们加班多不多'或'没什么想问的了'，前者容易被贴低意愿标签，后者显得对公司毫无研究。

### 2. 回答'你最大的缺点是什么'的黄金公式

真实但非致命的工程习惯 + 已经采取的具体改进措施 + 取得的积极变化（例如：'早期做方案时容易追求架构一步到位导致首版周期拉长，后来我强制自己先用 MVP 跑通端到端指标再灰度演进，星辰智脑 6 天就完成了全栈交付'）。

> **⚠️ 老陈避坑**：切忌说'我的缺点是太追求完美/太爱加班'（太假），也切忌暴露岗位致命伤（如'我不太擅长写代码'）。


## 💻 实操步骤 1：步骤一：实现技术面 + HR 面全真模拟加权评估打分器

- **目标文件**：`versions/day69/src/day69_mock_interview_scorer.py`
- **老陈代码旁白**：按照大厂 HC 评分标准对五大面试维度做加权统计与木桶红线校验，给出录用决策与薄弱项清单。


In [ ]:
"""
Day 69 - 模块一：技术面 + HR 面全真模拟评估打分器（大厂 HC 评分卡机制）
"""
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List

WEIGHTS = {
    "coding_algo": 0.25,
    "llm_rag_theory": 0.20,
    "project_depth": 0.25,
    "system_design": 0.15,
    "hr_culture_fit": 0.15,
}


@dataclass
class MockInterviewResult:
    candidate: str
    weighted_score: float
    decision: str  # STRONG_HIRE | HIRE | LEAN_NO_HIRE | NO_HIRE
    red_flags: List[str]
    weak_areas: List[str]
    scores: Dict[str, float]


def evaluate_mock_interview(candidate: str, scores: Dict[str, float]) -> MockInterviewResult:
    missing = [k for k in WEIGHTS if k not in scores]
    if missing:
        raise ValueError(f"缺少必填面试评估维度: {missing}")

    weighted = round(sum(scores[k] * w for k, w in WEIGHTS.items()), 2)
    red_flags = [k for k, v in scores.items() if v < 60.0]
    weak_areas = sorted([k for k, v in scores.items() if v < 80.0], key=lambda k: scores[k])

    if red_flags:
        decision = "NO_HIRE" if (len(red_flags) >= 2 or weighted < 70.0) else "LEAN_NO_HIRE"
    elif weighted >= 88.0:
        decision = "STRONG_HIRE"
    elif weighted >= 78.0:
        decision = "HIRE"
    else:
        decision = "LEAN_NO_HIRE"

    return MockInterviewResult(
        candidate=candidate,
        weighted_score=weighted,
        decision=decision,
        red_flags=red_flags,
        weak_areas=weak_areas,
        scores=dict(scores),
    )


if __name__ == "__main__":
    res = evaluate_mock_interview(
        "林同学",
        {
            "coding_algo": 84,
            "llm_rag_theory": 88,
            "project_depth": 90,
            "system_design": 72,
            "hr_culture_fit": 76,
        },
    )
    print("模拟面试评估结论:", res.decision, "总分:", res.weighted_score, "待提升项:", res.weak_areas)


### 🔍 步骤 1 代码深度解析

真实还原大厂 HC 的'加权均值 + 单项红线拦截'逻辑：哪怕总分 80+，只要某项跌破 60 分就会被 Bar Raiser 拦下。

**预期运行输出参考：**
```text
模拟面试评估结论: HIRE 总分: 83.3 待提升项: ['system_design', 'hr_culture_fit']
```


## 💻 实操步骤 2：步骤二：实现自动化 7 天靶向改进清单生成器

- **目标文件**：`versions/day69/src/day69_improvement_planner.py`
- **老陈代码旁白**：根据模拟面试结果中的薄弱维度与优先级，自动生成一份精准到天的 7 天冲刺整改行动表（Markdown）。


In [ ]:
"""
Day 69 - 模块二：基于模拟面试弱项的 7 天靶向改进清单生成器
"""
from __future__ import annotations

from typing import Dict, List

from src.day69_mock_interview_scorer import MockInterviewResult

REMEDIATION_PLAYBOOK: Dict[str, str] = {
    "coding_algo": "闭眼手撕 Attention/Top-P/令牌桶/TTL-LRU 与 5 道高频异步并发题，计时 25 分钟/题。",
    "llm_rag_theory": "复盘混合检索 RRF、Rerank、KV Cache、LoRA/QLoRA 数学直觉与防幻觉门禁。",
    "project_depth": "为星辰智脑补齐 4 张 5 层 ADR 防问穿卡片（含真实故障定位与量化对比）。",
    "system_design": "白板限时 30 分钟独立推导 10 万 DAU 知识库 Peak QPS、SQ8 向量内存与 GPU 算力账单。",
    "hr_culture_fit": "打磨 3 分钟自我介绍、动机一致性故事、缺点改进公式及 3 个高情商反问问题。",
}


def generate_7day_plan(result: MockInterviewResult) -> str:
    lines: List[str] = [
        f"# {result.candidate} 模拟面试 7 天靶向冲刺改进清单",
        f"- 当前综合分: **{result.weighted_score}** | 评估结论: **{result.decision}**",
        f"- 红线项(<60): {result.red_flags or '无'}",
        "",
        "## 每日靶向整改任务",
    ]
    targets = result.weak_areas or ["project_depth", "system_design"]
    for day_idx in range(1, 6):
        dim = targets[(day_idx - 1) % len(targets)]
        priority = "P0-红线攻坚" if dim in result.red_flags else "P1-专项拔高"
        lines.append(f"- **Day {day_idx} [{priority} - {dim}]**: {REMEDIATION_PLAYBOOK[dim]}")

    lines.append("- **Day 6 [全真二轮复测]**: 完整模拟 45m 技术面 + 20m HR 面，录屏回看语速与逻辑。")
    lines.append("- **Day 7 [投递冲刺启动]**: 按目标岗位 JD 微调简历关键词，开始首批重点投递。")
    return "\n".join(lines) + "\n"


if __name__ == "__main__":
    from src.day69_mock_interview_scorer import evaluate_mock_interview

    r = evaluate_mock_interview(
        "林同学",
        {
            "coding_algo": 55,
            "llm_rag_theory": 86,
            "project_depth": 88,
            "system_design": 74,
            "hr_culture_fit": 82,
        },
    )
    print(generate_7day_plan(r))


### 🔍 步骤 2 代码深度解析

将模拟面试结果直接转化为可执行、带优先级标记的 7 天行动清单，确保每一天都打在最提分的靶心上。

**预期运行输出参考：**
```text
# 林同学 模拟面试 7 天靶向冲刺改进清单
- 当前综合分: **76.35** | 评估结论: **LEAN_NO_HIRE**
- 红线项(<60): ['coding_algo']...
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day69/tests/test_day69.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 69 单元测试：模拟面试 HC 评估打分器与 7 天整改清单生成"""
import sys
from pathlib import Path
import pytest

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from src.day69_mock_interview_scorer import evaluate_mock_interview
from src.day69_improvement_planner import generate_7day_plan


def test_strong_hire_decision():
    res = evaluate_mock_interview(
        "张三",
        {"coding_algo": 92, "llm_rag_theory": 90, "project_depth": 94, "system_design": 86, "hr_culture_fit": 88},
    )
    assert res.decision == "STRONG_HIRE"
    assert res.red_flags == [] and res.weak_areas == []


def test_single_red_flag_blocks_hire():
    res = evaluate_mock_interview(
        "李四",
        {"coding_algo": 52, "llm_rag_theory": 90, "project_depth": 92, "system_design": 85, "hr_culture_fit": 85},
    )
    assert res.weighted_score >= 78.0
    assert res.red_flags == ["coding_algo"]
    assert res.decision == "LEAN_NO_HIRE"


def test_7day_plan_prioritizes_red_flag_and_missing_dim_raises():
    res = evaluate_mock_interview(
        "王五",
        {"coding_algo": 58, "llm_rag_theory": 85, "project_depth": 82, "system_design": 70, "hr_culture_fit": 85},
    )
    plan = generate_7day_plan(res)
    assert "P0-红线攻坚 - coding_algo" in plan
    assert "Day 7" in plan

    with pytest.raises(ValueError):
        evaluate_mock_interview("缺项", {"coding_algo": 80})

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：基础巩固：计算多次模拟面试的进步趋势（Score Delta）

**题目要求：**
编写 `calc_progress(round1_scores, round2_scores)` 返回每个维度的涨分幅度字典。

**思路提示：** 遍历键计算 `round(round2[k] - round1[k], 2)`。


In [ ]:
def calc_progress(r1: dict[str, float], r2: dict[str, float]) -> dict[str, float]:
    return {k: round(r2[k] - r1[k], 2) for k in r1}


diff = calc_progress({"coding": 58, "design": 70}, {"coding": 82, "design": 84})
assert diff == {"coding": 24, "design": 14}
print("二轮模拟提分统计通过:", diff)


**答案解析与评分标准：**

可视化两轮模拟的提分轨迹能极大增强求职冲刺期的自信心。


### 作业 2（企业实战）：企业实战：构建 Offer 薪资总包（Total Compensation）客观对比器

**题目要求：**
编写 `calc_annual_tc(base_monthly, months, signing_bonus=0, stock_annual=0)` 计算年化现金与总包，方便理性对比多个 Offer。

**思路提示：** 现金总计 = `base_monthly * months + signing_bonus`，总包 = 现金 + 股票年化折算。


In [ ]:
def calc_annual_tc(base_monthly: float, months: float, signing_bonus: float = 0.0, stock_annual: float = 0.0) -> dict:
    cash = round(base_monthly * months + signing_bonus, 2)
    return {"annual_cash": cash, "total_comp": round(cash + stock_annual, 2)}


o1 = calc_annual_tc(28000, 15, signing_bonus=20000, stock_annual=30000)
assert o1["annual_cash"] == 440000.0 and o1["total_comp"] == 470000.0
print("Offer 总包核算通过:", o1)


**答案解析与评分标准：**

对比 Offer 时要区分确定性更高的 Base 现金与浮动年终/期权，结合团队核心度综合决策。


### 作业 3（架构进阶）：架构进阶：实现面试题库错题间隔重复调度器（Spaced Repetition）

**题目要求：**
根据答题掌握度 `mastery`（1=未掌握, 2=模糊, 3=熟练），返回下次复习间隔天数（1天 / 3天 / 7天）。

**思路提示：** 字典映射 `{1: 1, 2: 3, 3: 7}`。


In [ ]:
def next_review_days(mastery: int) -> int:
    mapping = {1: 1, 2: 3, 3: 7}
    if mastery not in mapping:
        raise ValueError("mastery 仅支持 1/2/3")
    return mapping[mastery]


assert [next_review_days(1), next_review_days(2), next_review_days(3)] == [1, 3, 7]
print("艾宾浩斯面试复习调度通过")


**答案解析与评分标准：**

高频八股与手撕题采用间隔重复记忆法，比考前通宵死背效率高出 3 倍以上。


---

## 🎯 老陈收尾复盘

复盘完这场全真模拟面试，你对自己的强项和短板已经一清二楚，拿着这份 7 天靶向整改清单逐项打勾，真实面试拿到心仪 Offer 只是时间问题！明天就是第 70 天结业典礼，我们一起见证星辰智脑 v1.0.0 正式发布！
